# Split learning
### ResNet18 on CIFAR-100 + α = 0.1

In [ ]:
#=============================================================================
# Split learning: ResNet18 on CIFAR-100
# This program is Version1: Single program simulation 
# ============================================================================
import torch
from torch import nn
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
import math
import os.path
import pandas as pd

from sklearn.model_selection import train_test_split
from PIL import Image
from glob import glob 
from pandas import DataFrame
import random
import numpy as np
import os


import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import copy
import time

def _sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    print(torch.cuda.get_device_name(0))    

#===================================================================  
ALPHA = 0.1   # requested alpha
DATASET_NAME = "CIFAR100"
program = "SL - CIFAR-100 α=0.1"



print(f"---------{program}----------")              # this is to identify the program in the slurm outputs files

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# To print in color -------test/train of the client side
def prRed(skk): print("\033[91m {}\033[00m" .format(skk)) 
def prGreen(skk): print("\033[92m {}\033[00m" .format(skk))     

#===================================================================  
# No. of users
DATA_ROOT = "./data"
NUM_CLIENTS = 10
EPOCHS = 200
frac = 1   # participation of clients; if 1 then 100% clients participate in SL
BASE_LR = 0.0001
LOCAL_EPOCHS = 5
NUM_WORKERS = 4
BATCH_SIZE = 256
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4


CIFAR_MEAN = {
    "CIFAR10": [0.4914, 0.4822, 0.4465],
    "CIFAR100": [0.5071, 0.4867, 0.4408],
}[DATASET_NAME]
CIFAR_STD = {
    "CIFAR10": [0.2470, 0.2435, 0.2616],
    "CIFAR100": [0.2675, 0.2565, 0.2761],
}[DATASET_NAME]



# ====================== Confirm =======================================
cut_at = "Original at 2"
prRed(f"\tData:\t\t{DATASET_NAME}\n\tCut:\t\t{cut_at}\n\tGlobal Rounds:\t{EPOCHS}\n\tClients:\t{NUM_CLIENTS}\n\tLocal Epochs:\t{LOCAL_EPOCHS}\n\tα:\t\t{ALPHA}\n\tWarm-up:\t0\n\tRebuild:\t0")


#=====================================================================================================
#                           Client-side Model definition
#=====================================================================================================
# Model at client side
class ResNet18_client_side(nn.Module):
    def __init__(self):
        super(ResNet18_client_side, self).__init__()
        self.layer1 = nn.Sequential (
                nn.Conv2d(3, 64, kernel_size = 7, stride = 2, padding = 3, bias = False),
                nn.BatchNorm2d(64),
                nn.ReLU (inplace = True),
                nn.MaxPool2d(kernel_size = 3, stride = 2, padding =1),
            )
        self.layer2 = nn.Sequential  (
                nn.Conv2d(64, 64, kernel_size = 3, stride = 1, padding = 1, bias = False),
                nn.BatchNorm2d(64),
                nn.ReLU (inplace = True),
                nn.Conv2d(64, 64, kernel_size = 3, stride = 1, padding = 1),
                nn.BatchNorm2d(64),              
            )
        
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
        
        
    def forward(self, x):
        resudial1 = F.relu(self.layer1(x))
        out1 = self.layer2(resudial1)
        out1 = out1 + resudial1 # adding the resudial inputs -- downsampling not required in this layer
        resudial2 = F.relu(out1)
        return resudial2
 
 
           

net_glob_client = ResNet18_client_side()
if torch.cuda.device_count() > 1:
    print("We use",torch.cuda.device_count(), "GPUs")
    net_glob_client = nn.DataParallel(net_glob_client)   # to use the multiple GPUs; later we can change this to CPUs only 

net_glob_client.to(device)
print(net_glob_client)     


#=====================================================================================================
#                           Server-side Model definition
#=====================================================================================================
# Model at server side
class Baseblock(nn.Module):
    expansion = 1
    def __init__(self, input_planes, planes, stride = 1, dim_change = None):
        super(Baseblock, self).__init__()
        self.conv1 = nn.Conv2d(input_planes, planes, stride =  stride, kernel_size = 3, padding = 1)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, stride = 1, kernel_size = 3, padding = 1)
        self.bn2 = nn.BatchNorm2d(planes)
        self.dim_change = dim_change
        
    def forward(self, x):
        res = x
        output = F.relu(self.bn1(self.conv1(x)))
        output = self.bn2(self.conv2(output))
        
        if self.dim_change is not None:
            res =self.dim_change(res)
            
        output += res
        output = F.relu(output)
        
        return output


class ResNet18_server_side(nn.Module):
    def __init__(self, block, num_layers, classes):
        super(ResNet18_server_side, self).__init__()
        self.input_planes = 64
        self.layer3 = nn.Sequential (
                nn.Conv2d(64, 64, kernel_size = 3, stride = 1, padding = 1),
                nn.BatchNorm2d(64),
                nn.ReLU (inplace = True),
                nn.Conv2d(64, 64, kernel_size = 3, stride = 1, padding = 1),
                nn.BatchNorm2d(64),       
                )   
        
        self.layer4 = self._layer(block, 128, num_layers[0], stride = 2)
        self.layer5 = self._layer(block, 256, num_layers[1], stride = 2)
        self.layer6 = self._layer(block, 512, num_layers[2], stride = 2)
        self. averagePool = nn.AvgPool2d(kernel_size = 7, stride = 1)
        self.fc = nn.Linear(512 * block.expansion, classes)
        
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
        
        
    def _layer(self, block, planes, num_layers, stride = 2):
        dim_change = None
        if stride != 1 or planes != self.input_planes * block.expansion:
            dim_change = nn.Sequential(nn.Conv2d(self.input_planes, planes*block.expansion, kernel_size = 1, stride = stride),
                                       nn.BatchNorm2d(planes*block.expansion))
        netLayers = []
        netLayers.append(block(self.input_planes, planes, stride = stride, dim_change = dim_change))
        self.input_planes = planes * block.expansion
        for i in range(1, num_layers):
            netLayers.append(block(self.input_planes, planes))
            self.input_planes = planes * block.expansion
            
        return nn.Sequential(*netLayers)
        
    
    def forward(self, x):
        out2 = self.layer3(x)
        out2 = out2 + x          # adding the resudial inputs -- downsampling not required in this layer
        x3 = F.relu(out2)
        
        x4 = self. layer4(x3)
        x5 = self.layer5(x4)
        x6 = self.layer6(x5)
        
        # x7 = F.avg_pool2d(x6, 7)
        x7 = F.adaptive_avg_pool2d(x6, 1) 
        x8 = x7.view(x7.size(0), -1) 
        y_hat =self.fc(x8)
        
        return y_hat

# net_glob_server = ResNet18_server_side(Baseblock, [2,2,2], 7) #7 is my numbr of classes
if DATASET_NAME == "CIFAR10":
    net_glob_server = ResNet18_server_side(Baseblock, [2,2,2], 10)  # 10 classes for CIFAR-10
else:
    net_glob_server = ResNet18_server_side(Baseblock, [2,2,2], 100)


if torch.cuda.device_count() > 1:
    print("We use",torch.cuda.device_count(), "GPUs")
    net_glob_server = nn.DataParallel(net_glob_server)   # to use the multiple GPUs 

net_glob_server.to(device)
print(net_glob_server)      

#===================================================================================
# For Server Side Loss and Accuracy 
loss_train_collect = []
acc_train_collect = []
loss_test_collect = []
acc_test_collect = []
batch_acc_train = []
batch_loss_train = []
batch_acc_test = []
batch_loss_test = []
round_time_collect = [] # Stores seconds per global round

criterion = nn.CrossEntropyLoss()
count1 = 0
count2 = 0

#====================================================================================================
#                                  Server Side Program
#====================================================================================================
def calculate_accuracy(fx, y):
    preds = fx.max(1, keepdim=True)[1]
    correct = preds.eq(y.view_as(preds)).sum()
    acc = 100.00 *correct.float()/preds.shape[0]
    return acc

# to print train - test together in each round-- these are made global
acc_avg_all_user_train = 0
loss_avg_all_user_train = 0
loss_train_collect_user = []
acc_train_collect_user = []
loss_test_collect_user = []
acc_test_collect_user = []


#client idx collector
idx_collect = []
l_epoch_check = False
fed_check = False


optimizer_server = torch.optim.SGD(net_glob_server.parameters(), lr = BASE_LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

# Server-side function associated with Training 
def train_server(fx_client, y, l_epoch_count, l_epoch, idx, len_batch):
    global optimizer_server
    global net_glob_server, criterion, device, batch_acc_train, batch_loss_train, l_epoch_check, fed_check
    global loss_train_collect, acc_train_collect, count1, acc_avg_all_user_train, loss_avg_all_user_train, idx_collect
    global loss_train_collect_user, acc_train_collect_user
    
    net_glob_server.train()
    
    # Clear gradient
    optimizer_server.zero_grad() 
    
    fx_client = fx_client.to(device)
    y = y.to(device)
    
    #---------forward prop-------------
    fx_server = net_glob_server(fx_client)
    
    # calculate loss
    loss = criterion(fx_server, y)
    # calculate accuracy
    acc = calculate_accuracy(fx_server, y)
    
    #--------backward prop--------------
    loss.backward()
    dfx_client = fx_client.grad.clone().detach()
    optimizer_server.step()
    
    batch_loss_train.append(loss.item())
    batch_acc_train.append(acc.item())
    
    # server-side model net_glob_server is global so it is updated automatically in each pass to this function
        # count1: to track the completion of the local batch associated with one client
    count1 += 1
    if count1 == len_batch:
        acc_avg_train = sum(batch_acc_train)/len(batch_acc_train)           # it has accuracy for one batch
        loss_avg_train = sum(batch_loss_train)/len(batch_loss_train)
        
        batch_acc_train = []
        batch_loss_train = []
        count1 = 0
        
        prRed('Client{} Train => Local Epoch: {} \tAcc: {:.3f} \tLoss: {:.4f}'.format(idx, l_epoch_count, acc_avg_train, loss_avg_train))
        
                
        # If one local epoch is completed, after this a new client will come
        if l_epoch_count == l_epoch-1:
            
            l_epoch_check = True                # for evaluate_server function - to check local epoch has hitted 
                       
            # we store the last accuracy in the last batch of the epoch and it is not the average of all local epochs
            # this is because we work on the last trained model and its accuracy (not earlier cases)
            
            #print("accuracy = ", acc_avg_train)
            acc_avg_train_all = acc_avg_train
            loss_avg_train_all = loss_avg_train
                        
            # accumulate accuracy and loss for each new user
            loss_train_collect_user.append(loss_avg_train_all)
            acc_train_collect_user.append(acc_avg_train_all)
            
            # collect the id of each new user                        
            if idx not in idx_collect:
                idx_collect.append(idx) 
                #print(idx_collect)
        
        # This is to check if all users are served for one round --------------------
        if len(idx_collect) == NUM_CLIENTS:
            fed_check = True                                                  # for evaluate_server function  - to check fed check has hitted
            # all users served for one round ------------------------- output print and update is done in evaluate_server()
            # for nicer display 
                        
            idx_collect = []
            
            acc_avg_all_user_train = sum(acc_train_collect_user)/len(acc_train_collect_user)
            loss_avg_all_user_train = sum(loss_train_collect_user)/len(loss_train_collect_user)
            
            loss_train_collect.append(loss_avg_all_user_train)
            acc_train_collect.append(acc_avg_all_user_train)
            
            acc_train_collect_user = []
            loss_train_collect_user = []
            
    # send gradients to the client               
    return dfx_client

# Server-side functions associated with Testing
def evaluate_server(fx_client, y, idx, len_batch, ell):
    global net_glob_server, criterion, batch_acc_test, batch_loss_test
    global loss_test_collect, acc_test_collect, count2, num_users, acc_avg_train_all, loss_avg_train_all, l_epoch_check, fed_check
    global loss_test_collect_user, acc_test_collect_user, acc_avg_all_user_train, loss_avg_all_user_train
    
    net_glob_server.eval()
  
    with torch.no_grad():
        fx_client = fx_client.to(device)
        y = y.to(device) 
        #---------forward prop-------------
        fx_server = net_glob_server(fx_client)
        
        # calculate loss
        loss = criterion(fx_server, y)
        # calculate accuracy
        acc = calculate_accuracy(fx_server, y)
        
        
        batch_loss_test.append(loss.item())
        batch_acc_test.append(acc.item())
        
               
        count2 += 1
        if count2 == len_batch:
            acc_avg_test = sum(batch_acc_test)/len(batch_acc_test)
            loss_avg_test = sum(batch_loss_test)/len(batch_loss_test)
            
            batch_acc_test = []
            batch_loss_test = []
            count2 = 0
            
            prGreen('Client{} Test =>                   \tAcc: {:.3f} \tLoss: {:.4f}'.format(idx, acc_avg_test, loss_avg_test))
            
            # if a local epoch is completed   
            if l_epoch_check:
                l_epoch_check = False
                
                # Store the last accuracy and loss
                acc_avg_test_all = acc_avg_test
                loss_avg_test_all = loss_avg_test
                        
                loss_test_collect_user.append(loss_avg_test_all)
                acc_test_collect_user.append(acc_avg_test_all)
                
            # if all users are served for one round ----------                    
            if fed_check:
                fed_check = False
                                
                acc_avg_all_user = sum(acc_test_collect_user)/len(acc_test_collect_user)
                loss_avg_all_user = sum(loss_test_collect_user)/len(loss_test_collect_user)
            
                loss_test_collect.append(loss_avg_all_user)
                acc_test_collect.append(acc_avg_all_user)
                acc_test_collect_user = []
                loss_test_collect_user= []
                              
                print("====================== SERVER V1==========================")
                print(' Train: Round {:3d}, Avg Accuracy {:.3f} | Avg Loss {:.3f}'.format(ell, acc_avg_all_user_train, loss_avg_all_user_train))
                print(' Test: Round {:3d}, Avg Accuracy {:.3f} | Avg Loss {:.3f}'.format(ell, acc_avg_all_user, loss_avg_all_user))
                # print("==========================================================")
         
    return 

#==============================================================================================================
#                                       Clients Side Program
#==============================================================================================================
class DatasetSplit(Dataset):
    def __init__(self, dataset, idxs):
        self.dataset = dataset
        self.idxs = list(idxs)

    def __len__(self):
        return len(self.idxs)

    def __getitem__(self, item):
        image, label = self.dataset[self.idxs[item]]
        return image, label

# Client-side functions associated with Training and Testing
class Client(object):
    def __init__(self, net_client_model, idx, lr, device, dataset_train = None, dataset_test = None, idxs = None, idxs_test = None):
        self.idx = idx
        self.device = device
        self.lr = lr
        self.local_ep = LOCAL_EPOCHS 
        #self.selected_clients = []
        self.ldr_train = DataLoader(DatasetSplit(dataset_train, idxs), batch_size = BATCH_SIZE, shuffle = True, num_workers=NUM_WORKERS)
        self.ldr_test = DataLoader(DatasetSplit(dataset_test, idxs_test), batch_size = BATCH_SIZE, shuffle = False, num_workers=NUM_WORKERS)
        

    def train(self, net):
        net.train()
        optimizer_client = torch.optim.SGD(net.parameters(), lr = self.lr, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY) 
        
        for iter in range(self.local_ep):
            len_batch = len(self.ldr_train)
            for batch_idx, (images, labels) in enumerate(self.ldr_train):
                images, labels = images.to(self.device), labels.to(self.device)
                optimizer_client.zero_grad()
                #---------forward prop-------------
                fx = net(images)
                client_fx = fx.clone().detach().requires_grad_(True)
                
                # Sending activations to server and receiving gradients from server
                dfx = train_server(client_fx, labels, iter, self.local_ep, self.idx, len_batch)
                
                #--------backward prop -------------
                fx.backward(dfx)
                optimizer_client.step()
                            
            
            #prRed('Client{} Train => Epoch: {}'.format(self.idx, ell))
           
        return net.state_dict() 
    
    def evaluate(self, net, ell):
        net.eval()
           
        with torch.no_grad():
            len_batch = len(self.ldr_test)
            for batch_idx, (images, labels) in enumerate(self.ldr_test):
                images, labels = images.to(self.device), labels.to(self.device)
                #---------forward prop-------------
                fx = net(images)
                
                # Sending activations to server 
                evaluate_server(fx, labels, self.idx, len_batch, ell)
            
            #prRed('Client{} Test => Epoch: {}'.format(self.idx, ell))
            
        return          
#=====================================================================================================
# dataset_iid() will create a dictionary to collect the indices of the data samples randomly for each client
# IID HAM10000 datasets will be created based on this
def dataset_iid(dataset, num_users):
    
    num_items = int(len(dataset)/num_users)
    dict_users, all_idxs = {}, [i for i in range(len(dataset))]
    for i in range(num_users):
        dict_users[i] = set(np.random.choice(all_idxs, num_items, replace = False))
        all_idxs = list(set(all_idxs) - dict_users[i])
    return dict_users    

def dataset_noniid_dirichlet(dataset, num_users, alpha=ALPHA, seed=SEED):
    """Return dict_users: {client_id -> set(sample_indices)} using Dirichlet(α)."""
    rng = np.random.default_rng(seed)
    labels = np.array(dataset.targets)  # CIFAR-10 stores labels here
    n_classes = len(np.unique(labels))
    dict_users = {i: set() for i in range(num_users)}

    for c in range(n_classes):
        idxs = np.where(labels == c)[0]
        rng.shuffle(idxs)
        
        # draw proportions for this class
        p = rng.dirichlet([alpha] * num_users)
        
        # split indices by cumulative proportions
        cuts = (np.cumsum(p) * len(idxs)).astype(int)[:-1]
        
        splits = np.split(idxs, cuts)
        for client_id, split in enumerate(splits):
            dict_users[client_id].update(split.tolist())
    return dict_users




#=============================================================================
#                         Data loading 
#============================================================================= 

# CIFAR-10 normalization
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
])

if DATASET_NAME == "CIFAR10":
    dataset_train = datasets.CIFAR10(
        root=DATA_ROOT, train=True, download=True, transform=train_transforms
    )
    dataset_test = datasets.CIFAR10(
        root=DATA_ROOT, train=False, download=True, transform=test_transforms
    )
    NUM_CLASSES = 10
else:
    dataset_train = datasets.CIFAR100(
        root=DATA_ROOT, train=True, download=True, transform=train_transforms
    )
    dataset_test = datasets.CIFAR100(
        root=DATA_ROOT, train=False, download=True, transform=test_transforms
    )
    NUM_CLASSES = 100



#=============================================================================
#                         Forgetting metric setup 
#============================================================================= 

from collections import defaultdict

# Treat each global round as a window (quick baseline).
# If you want true distribution shifts, predefine your own windows and fill `val_loaders`.
T = EPOCHS

val_loaders = {}
# Deterministic, disjoint slices of the test set: window t gets items t, t+T, t+2T, ...
rng_idx = np.arange(len(dataset_test))
for t in range(T):
    idxs = rng_idx[t::T]
    val_loaders[t] = DataLoader(DatasetSplit(dataset_test, idxs),
                                batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,drop_last=False)

best_loss_per_window = defaultdict(lambda: float('inf'))  # stores min historical loss per window
forgetting_log = []  # per round dicts: {"round": t, "mean_forgetting": x, "per_window": {...}}
# ============================================================================

def _eval_loss_on_loader(model_client, model_server, loader, device, criterion):
    """Evaluate mean loss on a loader using split models."""
    # Handle DataParallel transparently
    mc = model_client.module if isinstance(model_client, nn.DataParallel) else model_client
    ms = model_server.module if isinstance(model_server, nn.DataParallel) else model_server
    mc.eval(); ms.eval()
    loss_sum, n = 0.0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            fx = mc(images)
            logits = ms(fx)
            loss = criterion(logits, labels)
            bs = labels.size(0)
            loss_sum += loss.item() * bs
            n += bs
    return loss_sum / max(1, n)

#----------------------------------------------------------------
# dict_users = dataset_iid(dataset_train, num_users)
# dict_users_test = dataset_iid(dataset_test, num_users)

dict_users      = dataset_noniid_dirichlet(dataset_train, num_users=NUM_CLIENTS, alpha=ALPHA)
dict_users_test = dataset_iid(dataset_test, num_users=NUM_CLIENTS)  



#net_glob_client.train()
# this epoch is global epoch, also known as rounds
# this epoch is global epoch, also known as rounds
for iter in range(EPOCHS):
    # ---- START TIMER ----
    _sync_cuda()
    t0 = time.perf_counter()
    # ---------------------

    m = max(int(frac * NUM_CLIENTS), 1)
    idxs_users = np.random.choice(range(NUM_CLIENTS), m, replace=False)

    # Sequential training/testing among clients
    for idx in idxs_users:
        local = Client(net_glob_client, idx, BASE_LR, device,
                       dataset_train=dataset_train, dataset_test=dataset_test,
                       idxs=dict_users[idx], idxs_test=dict_users_test[idx])

        # Training
        w_client = local.train(net=copy.deepcopy(net_glob_client).to(device))

        # Testing
        local.evaluate(net=copy.deepcopy(net_glob_client).to(device), ell=iter)

        # Update client weights for next client
         # copy weight to net_glob_client -- use to update the client-side model of the next client to be trained
        net_glob_client.load_state_dict(w_client)

    # ---- STOP TIMER & LOG ----
    _sync_cuda()
    dt = time.perf_counter() - t0
    
    # === Forgetting update at end of round `iter` ================================
    t = iter  # current window id

    # 1) Update the "best so far" for the CURRENT window using the current model
    L_cur_t = _eval_loss_on_loader(net_glob_client, net_glob_server, val_loaders[t], device, criterion)
    if math.isfinite(L_cur_t):
        best_loss_per_window[t] = min(best_loss_per_window[t], L_cur_t)

    # 2) Compute forgetting on all PAST windows t0 < t
    per_win_fgt = {}
    for t0 in range(t):
        L_now = _eval_loss_on_loader(net_glob_client, net_glob_server, val_loaders[t0], device, criterion)
        best = best_loss_per_window[t0]
        if not (math.isfinite(L_now) and math.isfinite(best)):
            continue
        Fgt = L_now - best
        per_win_fgt[t0] = Fgt
        
    mean_fgt = float(np.mean(list(per_win_fgt.values()))) if per_win_fgt else 0.0
    forgetting_log.append({"round": t, "mean_forgetting": mean_fgt, "per_window": per_win_fgt})
    print(f"Forgetting : Round {t}, mean_forgetting : {mean_fgt:.3f}")
    print("==========================================================")
    # =============================================================================
    
    round_time_collect.append(dt)
    # -------------------------- 
    



print("Training and Evaluation completed!")    

#===============================================================================
# Save output data to .excel file (we use for comparision plots)
# Align lengths safely (in case of early stop or mismatch)
n = min(len(acc_train_collect), len(acc_test_collect), len(round_time_collect), len(forgetting_log))
round_process = list(range(1, n+1))
mean_fgt_series = [d["mean_forgetting"] for d in forgetting_log[:n]]

df = DataFrame({
    'round': round_process,
    'acc_train': acc_train_collect[:n],
    'acc_test':  acc_test_collect[:n],
    'round_time_sec': round_time_collect[:n],   # time
    'mean_FgT_loss': mean_fgt_series, 
})

# write CSV — easier for scripting/comparison tools
file_name_csv = program + ".csv"
df.to_csv(file_name_csv, index=False)

print("Saved:", file_name_csv)

#=============================================================================
#                         Program Completed
#============================================================================= 

NVIDIA H200
---------SL - CIFAR-100 α=0.1----------
 	Data:		CIFAR100
	Cut:		Original at 2
	Global Rounds:	100
	Clients:	10
	Local Epochs:	5
	α:		0.1
	Warm-up:	0
	Rebuild:	0
ResNet18_client_side(
  (layer1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
)
ResNet18_server_side(
  (layer3): Sequential(
    (0): Conv2d(64, 64, kernel_size=(3, 3), s